# Pertemuan 10 — Praktikum 3: Klasifikasi Teks SMS dengan Naive Bayes

Praktikum ini mengklasifikasikan SMS `spam` dan `ham` menggunakan **Multinomial Naive Bayes** dengan fitur **Bag of Words** dari `CountVectorizer`.

Sumber: [Praktikum 3](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js09-klasifikasi/praktikum-3).

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd

## 2. Langkah 1 — Load Data

Data dimuat dengan `encoding='latin-1'` karena berkas tidak memakai UTF-8. Dataset memuat teks SMS beserta label `spam`/`ham`.

In [2]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## 3. Langkah 2a — Drop Kolom dan Ubah Nama Kolom

Tiga kolom terakhir yang kosong dibuang, kemudian kolom `v1` dan `v2` diganti namanya menjadi `Labels` dan `SMS`.

In [3]:
df = df.drop(df.iloc[:, 2:], axis=1)
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})
df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 4. Langkah 2b — Inspeksi Data

Jumlah data tiap kelas, kelengkapan data, dan statistik deskriptif diperiksa.

In [4]:
print(df['Labels'].value_counts())
print('\n')
print(df.info())
print('\n')
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Labels  5572 non-null   str  
 1   SMS     5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB
None


       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


## 5. Langkah 2c — Encoding Label

Label kategorik diubah menjadi angka: `spam` menjadi 1 dan `ham` menjadi 0.

In [5]:
new_labels = {'spam': 1, 'ham': 0}
df['Labels'] = df['Labels'].map(new_labels)
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


## 6. Langkah 2d — Pisahkan Fitur dengan Label

Fitur berupa teks SMS dan label berupa nilai hasil encoding dipisahkan.

In [6]:
X = df['SMS'].values
y = df['Labels'].values

## 7. Langkah 3 — Ekstraksi Fitur (Bag of Words)

Data dibagi menjadi latih dan uji terlebih dahulu untuk mencegah kebocoran informasi. `CountVectorizer` dipasang hanya pada data latih, lalu data uji hanya ditransformasikan.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

bow = CountVectorizer()
X_train = bow.fit_transform(X_train)
X_test = bow.transform(X_test)

print('Jumlah fitur:', len(bow.get_feature_names_out()))
print(f'Dimensi data latih: {X_train.shape}')
print(f'Dimensi data uji: {X_test.shape}')

Jumlah fitur: 7727
Dimensi data latih: (4457, 7727)
Dimensi data uji: (1115, 7727)


## 8. Langkah 4 — Training dan Evaluasi Model

Model Multinomial Naive Bayes dilatih dan dievaluasi dengan akurasi pada data latih dan data uji.

In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

mnb = MultinomialNB()
mnb.fit(X_train, y_train)

acc_train = accuracy_score(y_train, mnb.predict(X_train))
acc_test = accuracy_score(y_test, mnb.predict(X_test))

print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9775784753363229


## 9. Kesimpulan Praktikum 3

Multinomial Naive Bayes dengan Bag of Words berhasil mengklasifikasikan SMS spam dengan akurasi tinggi karena model ini memang dirancang untuk data hitungan seperti frekuensi kata.